# 04 - Statistical Analysis

**Owner:** Analysis Lead  
**Goal:** Test whether the EDA patterns are statistically meaningful and convert results into decision-ready evidence.

This notebook uses `scipy` by default and uses `statsmodels` for regression if available. In Colab, run `%pip install statsmodels` if needed.

In [ ]:
from pathlib import Path
import itertools
import warnings

import numpy as np
import pandas as pd
from scipy import stats

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
ANALYSIS_DIR = PROCESSED_DIR / 'analysis'
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

try:
    import statsmodels.formula.api as smf
    HAS_STATSMODELS = True
except Exception as exc:
    HAS_STATSMODELS = False
    print('statsmodels unavailable. Regression section will be skipped.')
    print(type(exc).__name__, exc)

In [ ]:
cleaned_paths = sorted(PROCESSED_DIR.glob('cleaned_hospital_*.csv'))
frames = [pd.read_csv(path) for path in cleaned_paths]
df = pd.concat(frames, ignore_index=True)
print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns')
display(df.head())

## 1. Helper Functions

Use consistent test summaries so the report can quote p-values, effect sizes, and business interpretation clearly.

In [ ]:
def cramers_v(table):
    chi2, _, _, _ = stats.chi2_contingency(table)
    n = table.to_numpy().sum()
    r, k = table.shape
    denom = n * (min(k - 1, r - 1))
    return np.sqrt(chi2 / denom) if denom else np.nan


def eta_squared_anova(groups):
    values = np.concatenate(groups)
    grand_mean = np.mean(values)
    ss_between = sum(len(g) * (np.mean(g) - grand_mean) ** 2 for g in groups)
    ss_total = sum((x - grand_mean) ** 2 for x in values)
    return ss_between / ss_total if ss_total else np.nan


def cost_group_test(group_col):
    groups = [g['Treatment_Cost'].dropna().to_numpy() for _, g in df.groupby(group_col)]
    labels = [str(name) for name, _ in df.groupby(group_col)]
    f_stat, anova_p = stats.f_oneway(*groups)
    h_stat, kruskal_p = stats.kruskal(*groups)
    return {
        'test_area': 'Treatment_Cost',
        'factor': group_col,
        'groups': ', '.join(labels),
        'test_used': 'ANOVA and Kruskal-Wallis',
        'statistic': round(float(h_stat), 4),
        'p_value': float(kruskal_p),
        'effect_size': round(float(eta_squared_anova(groups)), 4),
        'interpretation': 'Treatment cost differs significantly by ' + group_col if kruskal_p < 0.05 else 'No statistically significant treatment-cost difference by ' + group_col,
        'secondary_result': f'ANOVA F={f_stat:.4f}, p={anova_p:.4g}',
    }


def chi_square_test(row_col, col_col='Outcome'):
    table = pd.crosstab(df[row_col], df[col_col])
    chi2, p_value, dof, expected = stats.chi2_contingency(table)
    cv = cramers_v(table)
    return {
        'test_area': col_col,
        'factor': row_col,
        'groups': ', '.join(map(str, table.index.tolist())),
        'test_used': 'Chi-square independence',
        'statistic': round(float(chi2), 4),
        'p_value': float(p_value),
        'effect_size': round(float(cv), 4),
        'interpretation': col_col + ' is associated with ' + row_col if p_value < 0.05 else col_col + ' is not significantly associated with ' + row_col,
        'secondary_result': f'dof={dof}',
    }


def spearman_test(x_col, y_col='Treatment_Cost'):
    rho, p_value = stats.spearmanr(df[x_col], df[y_col], nan_policy='omit')
    return {
        'test_area': y_col,
        'factor': x_col,
        'groups': 'numeric correlation',
        'test_used': 'Spearman correlation',
        'statistic': round(float(rho), 4),
        'p_value': float(p_value),
        'effect_size': round(abs(float(rho)), 4),
        'interpretation': f'{x_col} has a statistically significant monotonic relationship with {y_col}' if p_value < 0.05 else f'{x_col} has no statistically significant monotonic relationship with {y_col}',
        'secondary_result': 'rho reported as statistic',
    }


def p_label(p):
    if p < 0.001:
        return '<0.001'
    return f'{p:.4f}'

## 2. Cost Difference Tests

Hypotheses:

- H1: Treatment cost differs by diagnosis.
- H2: Treatment cost differs by hospital.
- H3: Treatment cost differs by hospital type, age group, economic status, and insurance status.

Kruskal-Wallis is used as the primary result because treatment costs are usually skewed.

In [ ]:
cost_tests = pd.DataFrame([
    cost_group_test('Diagnosis'),
    cost_group_test('Hospital'),
    cost_group_test('Hospital_Type'),
    cost_group_test('Age_Group'),
    cost_group_test('Economic_Status'),
    cost_group_test('Insurance'),
])

cost_tests['p_value_label'] = cost_tests['p_value'].map(p_label)
display(cost_tests[['test_area', 'factor', 'test_used', 'statistic', 'p_value_label', 'effect_size', 'interpretation', 'secondary_result']])

In [ ]:
# Pairwise diagnosis comparisons for treatment cost.
# This helps identify which categories create the strongest practical differences.
pairwise_rows = []
for a, b in itertools.combinations(sorted(df['Diagnosis'].dropna().unique()), 2):
    x = df.loc[df['Diagnosis'] == a, 'Treatment_Cost']
    y = df.loc[df['Diagnosis'] == b, 'Treatment_Cost']
    stat, p_value = stats.mannwhitneyu(x, y, alternative='two-sided')
    pairwise_rows.append({
        'group_a': a,
        'group_b': b,
        'median_a': round(x.median(), 2),
        'median_b': round(y.median(), 2),
        'median_difference_a_minus_b': round(x.median() - y.median(), 2),
        'p_value': p_value,
    })

pairwise_diagnosis_cost = pd.DataFrame(pairwise_rows).sort_values('p_value')
pairwise_diagnosis_cost['p_value_label'] = pairwise_diagnosis_cost['p_value'].map(p_label)
pairwise_diagnosis_cost.to_csv(ANALYSIS_DIR / 'pairwise_diagnosis_cost_tests.csv', index=False)
display(pairwise_diagnosis_cost)

## 3. Outcome Association Tests

Hypotheses:

- H4: Patient outcome is associated with diagnosis.
- H5: Patient outcome is associated with age group, economic status, insurance, hospital, and region.

Use Cramer's V as the effect size for categorical association. Higher values mean a stronger relationship.

In [ ]:
outcome_tests = pd.DataFrame([
    chi_square_test('Diagnosis'),
    chi_square_test('Age_Group'),
    chi_square_test('Economic_Status'),
    chi_square_test('Insurance'),
    chi_square_test('Hospital'),
    chi_square_test('Region'),
])

outcome_tests['p_value_label'] = outcome_tests['p_value'].map(p_label)
display(outcome_tests[['test_area', 'factor', 'test_used', 'statistic', 'p_value_label', 'effect_size', 'interpretation', 'secondary_result']])

In [ ]:
for factor in ['Diagnosis', 'Age_Group', 'Economic_Status', 'Insurance', 'Hospital', 'Region']:
    print()
    print('Outcome mix by', factor)
    display(pd.crosstab(df[factor], df['Outcome'], normalize='index').mul(100).round(2))


## 4. Operational Correlations

Hypotheses:

- H6: Treatment cost is related to doctor availability, cleanliness score, doctor experience, and age.
- H7: Operational quality metrics may move together and should be monitored together in Tableau.

In [ ]:
correlation_tests = pd.DataFrame([
    spearman_test('Age'),
    spearman_test('Doctor_Experience_Years'),
    spearman_test('Doctor_Availability'),
    spearman_test('Cleanliness_Score'),
])
correlation_tests['p_value_label'] = correlation_tests['p_value'].map(p_label)
display(correlation_tests[['test_area', 'factor', 'test_used', 'statistic', 'p_value_label', 'effect_size', 'interpretation']])

ops_corr = df[['Treatment_Cost', 'Age', 'Doctor_Experience_Years', 'Doctor_Availability', 'Cleanliness_Score']].corr(method='spearman').round(4)
display(ops_corr)

## 5. Optional Regression Model

Use this when `statsmodels` is available. It estimates treatment cost while controlling for hospital, diagnosis, patient segment, and operational factors.

This should not replace the simpler tests above; it supports a stronger explanation for the report.

In [ ]:
if HAS_STATSMODELS:
    model_df = df.copy()
    formula = 'Treatment_Cost ~ Age + Doctor_Experience_Years + Doctor_Availability + Cleanliness_Score + C(Hospital) + C(Diagnosis) + C(Insurance) + C(Economic_Status) + C(Age_Group)'
    model = smf.ols(formula, data=model_df).fit(cov_type='HC3')
    print(model.summary())

    coef_table = pd.DataFrame({
        'term': model.params.index,
        'coefficient': model.params.values,
        'p_value': model.pvalues.values,
        'conf_low': model.conf_int()[0].values,
        'conf_high': model.conf_int()[1].values,
    }).sort_values('p_value')
    coef_table.to_csv(ANALYSIS_DIR / 'ols_treatment_cost_coefficients.csv', index=False)
    display(coef_table.head(20))
else:
    print('Skipped regression because statsmodels is not installed in this environment.')

## 6. Statistical Evidence Summary

This table is the main handoff to the Strategy Lead and PPT & Quality Lead. Include statistically significant, practically meaningful findings in the final report.

In [ ]:
all_tests = pd.concat([cost_tests, outcome_tests, correlation_tests], ignore_index=True)
all_tests['p_value_label'] = all_tests['p_value'].map(p_label)
all_tests['significant_at_0_05'] = all_tests['p_value'] < 0.05
all_tests = all_tests.sort_values(['significant_at_0_05', 'effect_size'], ascending=[False, False])
all_tests.to_csv(ANALYSIS_DIR / 'statistical_tests_summary.csv', index=False)

display(all_tests[['test_area', 'factor', 'test_used', 'p_value_label', 'effect_size', 'significant_at_0_05', 'interpretation']])

In [ ]:
decision_readout = []
for _, row in all_tests.iterrows():
    if row['significant_at_0_05']:
        decision_readout.append({
            'finding': row['interpretation'],
            'evidence': f"{row['test_used']}: statistic={row['statistic']}, p={p_label(row['p_value'])}, effect size={row['effect_size']}",
            'business_use': 'Use as evidence for dashboard focus, recommendations, or report narrative.'
        })

decision_readout = pd.DataFrame(decision_readout)
decision_readout.to_csv(ANALYSIS_DIR / 'statistical_decision_readout.csv', index=False)
display(decision_readout)